In [1]:
import pandas as pd
from scipy.stats import spearmanr
import numpy as np

In [2]:
CYV_PATH = "country_year_women_ratio_vdem.csv"

cyv = pd.read_csv(CYV_PATH)

# sample restriction 
post = cyv[cyv["year"] >= 2000].copy()

# Keeping only valid rows for analysis
post = post.dropna(subset=["female_ratio", "female_total", "people_total", "v2x_polyarchy"]).copy()
post = post[post["people_total"] > 0].copy()

# one row per country
country = (post.groupby("country", as_index=False)
             .agg(
                 female_total=("female_total","sum"),
                 people_total=("people_total","sum"),
                 n_years=("year","nunique"),
                 n_rows=("year","size"),
                 n_images_people=("n_images_people","sum"),
                 v2x_polyarchy_mean=("v2x_polyarchy","mean"),
             ))

country["female_ratio_country"] = country["female_total"] / country["people_total"]

# Spearman hypothesis test
rho, p = spearmanr(country["female_ratio_country"], country["v2x_polyarchy_mean"])

print("Country-level Spearman rho:", rho)
print("p-value:", p)
print("N (countries):", len(country))


Saved: country_level_post2000_for_test.csv
Country-level Spearman rho: 0.4960551646889985
p-value: 7.096206998584445e-10
N (countries): 137


In [3]:
df = pd.read_csv("country_level_post2000_for_test.csv")

x = df["female_ratio_country"].to_numpy()
y = df["v2x_polyarchy_mean"].to_numpy()

# Observed Spearman rank correlation
rho_obs, _ = spearmanr(x, y)

# permutation test
rng = np.random.default_rng(42)
B = 10000    #number of permutations
rhos = np.empty(B)

for b in range(B):
    y_perm = rng.permutation(y)
    rhos[b], _ = spearmanr(x, y_perm) #computing rho under permutation

p_perm = (np.sum(np.abs(rhos) >= np.abs(rho_obs)) + 1) / (B + 1) #avoid zero p-value

print("Observed rho:", rho_obs)
print("Permutation p-value:", p_perm)


Observed rho: 0.4960551646889985
Permutation p-value: 9.999000099990002e-05
